# 03 - Realistic Backtest, Benchmark and Lessons Learned

Turns the model's test-period signals into simulated trades with **transaction fees + slippage**, compares to **Buy & Hold SPY**,
and records the lessons. Run `02_modeling.ipynb` first (it saves `outputs/test_predictions.csv`), or run `python run_pipeline.py`.

**Execution assumptions (no look-ahead in execution either):** signal decided at the close of day *t* -> order filled at the **open of
day t+1**; each fill pays commission and slippage; long-only (Buy -> long, Sell -> cash, Hold -> keep position); no leverage.

In [ ]:
%matplotlib inline
import os, sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.width", 140); pd.set_option("display.max_columns", 40)

from spy_signals import Config, plotting as P
from spy_signals.data import load_ohlcv

# SPY_FAST=1 shrinks the hyper-parameter grids; SPY_DATA_SOURCE=synthetic runs offline (smoke test only!)
cfg = Config(fast=os.environ.get("SPY_FAST") == "1")
ohlcv = load_ohlcv(cfg)

In [ ]:
from spy_signals import backtest as bt
pred_file = Path(cfg.output_dir) / "test_predictions.csv"
assert pred_file.exists(), "Run notebooks/02_modeling.ipynb (or run_pipeline.py) first."
pr = pd.read_csv(pred_file, index_col=0, parse_dates=True)
print(f"{len(pr)} signal days: {pr.index.min().date()} -> {pr.index.max().date()}")
pr["pred"].map({-1: 'Sell', 0: 'Hold', 1: 'Buy'}).value_counts()

## 1. From signals to positions
* **Stateful** (the sensible design): Buy -> long, Sell -> cash, Hold -> keep what you have. Trades only when the view changes.
* **Daily Buy-only** (the cautionary design): long only on days the model says Buy. Flickering signals -> many round trips.

In [ ]:
desired = {
    "Model (stateful)": bt.signals_to_positions(pr["pred"], pr["confidence"], cfg.min_confidence, "stateful"),
    "Model (daily Buy-only)": bt.signals_to_positions(pr["pred"], pr["confidence"], cfg.min_confidence, "daily"),
}
results = {n: bt.run_backtest(ohlcv, p, n, cfg.fee, cfg.slippage, cfg.init_cash, cfg.engine) for n, p in desired.items()}
bh = bt.buy_and_hold(ohlcv, pr.index, cfg.fee, cfg.slippage, cfg.init_cash, cfg.engine)
print("engine:", bh.engine, f"| fee {cfg.fee:.2%} + slippage {cfg.slippage:.2%} per order")
perf = pd.DataFrame([bh.metrics()] + [r.metrics() for r in results.values()]).set_index("strategy")
perf.round(4)

## 2. Benchmark comparison: model portfolio vs Buy & Hold (required plot)

In [ ]:
P.plot_equity([bh, *results.values()]);

## 3. The transaction-cost lesson: re-run the same signals under different commissions

In [ ]:
sens = bt.fee_sensitivity(ohlcv, desired, [0.0, 0.0005, 0.001, 0.002, 0.005], cfg.slippage, cfg.init_cash, cfg.engine)
display(sens.pivot(index="fee", columns="strategy", values="final_value").round(0))
P.plot_fee_sensitivity(sens);

## 4. Library cross-check
If `vectorbt` is installed, its portfolio is compared with the transparent built-in engine (same timing/cost conventions). Small differences are expected from floating-point/rounding conventions; large ones mean a modelling difference worth investigating.

In [ ]:
if bh.engine == "vectorbt":
    ref = bt.run_backtest(ohlcv, desired["Model (stateful)"], "ref", cfg.fee, cfg.slippage, cfg.init_cash, "builtin")
    v = results["Model (stateful)"].equity.iloc[-1]; b = ref.equity.iloc[-1]
    print(f"vectorbt final ${v:,.2f} | built-in final ${b:,.2f} | relative diff {abs(v-b)/b:.4%}")
    display(results["Model (stateful)"].raw.stats())
else:
    print("vectorbt not installed in this environment - install it (pip install -r requirements.txt) to satisfy the library requirement.")

## 5. Auto-generated Lessons Learned (from `python run_pipeline.py`)

In [ ]:
rep = Path(cfg.output_dir) / "lessons_learned.md"
from IPython.display import Markdown
Markdown(rep.read_text()) if rep.exists() else print("Run `python run_pipeline.py` to generate outputs/lessons_learned.md")

## 6. Your Lessons Learned (write this in your own words - the grade is for the interpretation)
Answer with *your* numbers from above:

1. **Accuracy vs usefulness.** Did the model beat the majority baseline on balanced accuracy? Does the accuracy resemble the 35-40 % the assignment calls normal?
2. **Where did it fail?** Which class had the lowest recall (see the confusion matrix) and why might that be?
3. **Why did it under/outperform Buy & Hold?** Consider exposure (time in cash), number of orders, fee drag (fee-sensitivity table), and the single-window caveat.
4. **What does the fee table say about a model that trades every day?**
5. **Why is financial prediction hard?** Low signal-to-noise, non-stationarity, overlapping labels, efficient-market pressure.
6. **What would you try next, and how would you validate it without leaking?** (walk-forward refits, exogenous features such as VIX/yields, confidence filters, minimum holding period.)